In [16]:
import pandas as pd
import os
import re
from sqlalchemy import create_engine
from sqlalchemy import text
import gc

In [2]:
DB_USER = 'someuser'
DB_PASSWORD = 'somepassword'
DB_HOST = 'localhost'
DB_PORT = '3306'
DB_NAME = 'engineering_db'

In [3]:
# Створення підключення (engine) через SQLAlchemy
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [4]:
#Завантаження сирих даних
home = os.path.expanduser("~")
path_to_data = os.path.join(home, "Downloads")

In [ ]:
# csv у датафрейм:
df_campaigns_raw = pd.read_csv(os.path.join(path_to_data, "campaigns.csv"))
df_users_raw = pd.read_csv(os.path.join(path_to_data, "users.csv"))
df_events_raw = pd.read_csv(os.path.join(path_to_data, "events_header.csv"),nrows=1000000)

In [6]:
#унікальні країни з таблиць користувачів та кампаній
unique_countries = df_users_raw['Location'].unique()

In [7]:
#DataFrame для таблиці locations
df_locations = pd.DataFrame({'country_name': unique_countries})
# Додаємо ID (починаючи з 1)
df_locations.insert(0, 'location_id', range(1, 1 + len(df_locations)))

In [23]:
#DataFrame для таблиці advertisers
unique_advertisers = df_campaigns_raw['AdvertiserName'].unique()
df_advertisers = pd.DataFrame({'advertiser_name': unique_advertisers})
df_advertisers.insert(0, 'advertiser_id', range(1, 1 + len(df_advertisers)))

In [9]:
#Function for: Парсинг таргетингу кампаній
def parse_targeting(targeting_str):
    """ Функція для розбиття рядка таргетингу на окремі атрибути """
    try:
        # Пошук вікового діапазону (наприклад, Age 18-35)
        age_match = re.search(r'Age (\d+)-(\d+)', targeting_str)
        age_min = int(age_match.group(1)) if age_match else None
        age_max = int(age_match.group(2)) if age_match else None

        # Розподіл за комами для отримання інтересу та країни
        parts = [p.strip() for p in targeting_str.split(',')]
        interest = parts[1] if len(parts) > 1 else None
        country = parts[2] if len(parts) > 2 else None

        return pd.Series([age_min, age_max, interest, country])
    except:
        return pd.Series([None, None, None, None])

In [10]:
# парсинг таргетингу
df_campaigns_raw[['target_age_min', 'target_age_max', 'target_interest', 'target_country']] = \
    df_campaigns_raw['TargetingCriteria'].apply(parse_targeting)

In [ ]:
#завантажуємо незалежні довідники
df_locations.to_sql('locations', con=engine, if_exists='append', index=False)
df_advertisers.to_sql('advertisers', con=engine, if_exists='append', index=False)

In [25]:
df_advertisers.to_sql('advertisers', con=engine, if_exists='append', index=False)

100

In [26]:
# 1. Створюємо робочу копію, щоб не пошкодити сирі дані
df_campaigns = df_campaigns_raw.copy()

# 2. Приєднуємо advertiser_id
df_campaigns = df_campaigns.merge(df_advertisers, left_on='AdvertiserName', right_on='advertiser_name', how='left')

# 3. Приєднуємо location_id для країни таргетингу
df_campaigns = df_campaigns.merge(df_locations, left_on='target_country', right_on='country_name', how='left')

# 4. Перейменовуємо отриманий location_id на target_location_id (так вимагає схема БД)
df_campaigns.rename(columns={'location_id': 'target_location_id'}, inplace=True)

# 5. Очищення: видаляємо текстові колонки, які базі більше не потрібні
columns_to_drop = ['AdvertiserName', 'advertiser_name', 'TargetingCriteria', 'target_country', 'country_name']

# Видаляємо лише ті колонки, які дійсно є в датафреймі, щоб уникнути помилок
df_campaigns.drop(columns=[col for col in columns_to_drop if col in df_campaigns.columns], inplace=True)

print("Мапінг кампаній завершено! Таблиця готова до SQL.")
display(df_campaigns.head())

Мапінг кампаній завершено! Таблиця готова до SQL.


,CampaignID,CampaignName,CampaignStartDate,CampaignEndDate,AdSlotSize,Budget,RemainingBudget,target_age_min,target_age_max,target_interest,advertiser_id,target_location_id
0,1,Campaign_1,2024-10-05,2024-11-16,728x90,183204.20,183204.20,24,42,Gaming,1,5
1,2,Campaign_2,2024-10-03,2024-11-13,160x600,81431.52,81431.52,23,44,Sports,1,4
2,3,Campaign_3,2024-10-18,2024-11-04,300x250,276829.07,276829.07,22,34,Finance,1,2
3,4,Campaign_4,2024-10-04,2024-11-08,300x250,363466.39,363466.39,24,45,Health,1,3
4,5,Campaign_5,2024-10-06,2024-10-29,300x250,468699.97,468699.97,19,38,Health,1,2


In [30]:
df_campaigns.rename(columns={
    'CampaignID': 'campaign_id',
    'CampaignName': 'campaign_name',
    'CampaignStartDate': 'start_date',
    'CampaignEndDate': 'end_date',
    'Budget': 'budget',
    'AdSlotSize': 'ad_slot_size',
    'RemainingBudget': 'remaining_budget'
}, inplace=True, errors='ignore')

# Очищуємо таблицю перед завантаженням
with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE campaigns;"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

# Завантажуємо всі дані разом з колонками
df_campaigns.to_sql('campaigns', con=engine, if_exists='append', index=False)

print("Таблиця campaigns успішно завантажена")

Таблиця campaigns успішно завантажена без жодної втрати даних!


In [35]:
# Робимо копію сирих даних
df_users = df_users_raw.copy()

# Визначаємо точні назви колонок у CSV
loc_col = 'Location' if 'Location' in df_users.columns else 'location'
int_col = 'Interests' if 'Interests' in df_users.columns else 'interests'

# Мапінг location_id (приєднуємо ID країни з довідника)
df_users = df_users.merge(df_locations, left_on=loc_col, right_on='country_name', how='left')

# Перейменовуємо колонки
df_users.rename(columns={
    'UserID': 'user_id',
    'Age': 'age',
    'Gender': 'gender',
    'SignupDate': 'signup_date',       # Виправив: тепер як у базі!
    'RegistrationDate': 'signup_date'  # Про всяк випадок
}, inplace=True, errors='ignore')


if int_col in df_users.columns:
    df_user_interests_raw = df_users[['user_id', int_col]].copy()
else:
    df_user_interests_raw = None

# Очищення: видаляємо текстові колонки та інтереси з основної таблиці users
columns_to_drop = [loc_col, 'country_name', int_col]
df_users.drop(columns=[col for col in columns_to_drop if col in df_users.columns], inplace=True, errors='ignore')

# Завантаження в базу
with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE users;"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

df_users.to_sql('users', con=engine, if_exists='append', index=False)

print("Таблиця users успішно завантажена!")
display(df_users.head(3))

Таблиця users успішно завантажена!


,user_id,age,gender,signup_date,location_id
0,1,58,Male,2020-07-28,1
1,2,61,Male,2021-04-21,2
2,3,50,Male,2022-07-08,2


In [18]:
df_events_raw = pd.read_csv(os.path.join(path_to_data, "events_header.csv"),nrows=10)
df_events_raw.head(3)

,EventID,AdvertiserName,CampaignName,CampaignStartDate,CampaignEndDate,CampaignTargetingCriteria,CampaignTargetingInterest,CampaignTargetingCountry,AdSlotSize,UserID,Device,Location,Timestamp,BidAmount,AdCost,WasClicked,ClickTimestamp,AdRevenue,Budget,RemainingBudget
0,e067dae7-9aaa-4b9c-9195-7af63ce89216,Advertiser_30,Campaign_278,2024-10-10,2024-11-12,Age 29-46,Gaming,Germany,300x250,289903,Mobile,USA,2024-10-31T06:56:39,4.60,3.70,False,NaN,0.0,279306.60,279306.60
1,ea95edfa-1ba0-444d-a40d-a3f064dd5791,Advertiser_25,Campaign_235,2024-10-24,2024-11-28,Age 22-33,Gaming,Australia,728x90,145276,Desktop,UK,2024-11-13T09:19:34,3.50,2.85,False,NaN,0.0,88914.60,88914.60
2,5f12c537-ac88-4fcc-8d23-b3fa2b538a3b,Advertiser_95,Campaign_947,2024-10-31,2024-11-11,Age 27-42,Health,USA,300x250,83608,Desktop,USA,2024-11-08T14:36:14,3.75,3.16,False,NaN,0.0,207892.45,207892.45


In [19]:
df_events_raw = pd.read_csv(os.path.join(path_to_data, "events_header.csv"),nrows=1000000)

In [23]:
user = "user"
password = "userpassword"
host = "localhost" # или 127.0.0.1
db_name = "engineering_db"

engine = create_engine(f"mysql+pymysql://{user}:{password}@{host}:3306/{db_name}?local_infile=1")

print("Завантаження довідників з бази даних...")
with engine.connect() as conn:
    df_camp = pd.read_sql("SELECT campaign_id, campaign_name FROM campaigns", conn)
    df_loc = pd.read_sql("SELECT location_id, country_name FROM locations", conn)

# Читання файлу
print("Читання 1 000 000 рядків з подій...")
df_events = pd.read_csv(os.path.join(path_to_data, "events_header.csv"), nrows=1000000)
df_events.columns = df_events.columns.str.strip() # страховка від прихованих пробілів у назвах

# Трансформація: Заміна назв на ID (Merge робимо ОДИН раз)
print("Виконання merge для отримання чистих INT ID...")

# Мерджимо кампанії
df_events = df_events.merge(df_camp, left_on='CampaignName', right_on='campaign_name', how='left')
# Мерджимо локації
df_events = df_events.merge(df_loc, left_on='Location', right_on='country_name', how='left')

# Перейменування та очищення колонок під структуру SQL
df_events.rename(columns={
    'EventID': 'event_id',
    'UserID': 'user_id',
    'Device': 'device',
    'Timestamp': 'event_timestamp',
    'BidAmount': 'bid_amount',
    'AdCost': 'ad_cost',
    'WasClicked': 'was_clicked',
    'ClickTimestamp': 'click_timestamp',
    'AdRevenue': 'ad_revenue'
}, inplace=True, errors='ignore')

# Список колонок згідно з CREATE TABLE
target_cols = [
    'event_id', 'campaign_id', 'user_id', 'device', 'location_id',
    'event_timestamp', 'bid_amount', 'ad_cost', 'was_clicked',
    'click_timestamp', 'ad_revenue'
]

# Залишаємо колонки та обробляємо порожні значення
df_events = df_events[[c for c in target_cols if c in df_events.columns]].copy()
df_events['campaign_id'] = df_events['campaign_id'].fillna(0).astype(int)
df_events['location_id'] = df_events['location_id'].fillna(0).astype(int)

# Створення тимчасового файлу для LOAD DATA
temp_path = os.path.abspath("final_mapped_events.csv")
df_events.to_csv(temp_path, index=False, header=False)

# Швидке завантаження в MySQL
col_list = ", ".join(df_events.columns)
load_sql = f"""
LOAD DATA LOCAL INFILE '{temp_path.replace('\\', '/')}'
INTO TABLE events
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\\n'
({col_list});
"""

print("Старт завантаження в MySQL через LOAD DATA LOCAL INFILE...")
with engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE events;"))
    conn.execute(text(load_sql))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

if os.path.exists(temp_path):
    os.remove(temp_path)

print("Успішно завантажено дані")
gc.collect()

Завантаження довідників з бази даних...
Читання 1 000 000 рядків з подій...
Виконання merge для отримання чистих INT ID...
Старт завантаження в MySQL через LOAD DATA LOCAL INFILE...
Успішно завантажено дані


16695

In [24]:
engine.dispose()